# 02 Menjelajahi pola dan kualitas data

**Tujuan:** melihat distribusi limbah, format data bermasalah, dan hubungan antarvariabel. EDA yang memengaruhi keputusan model menggunakan training saja. Data validasi/test tetap tersimpan untuk evaluasi.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
while not (ROOT / 'src/experiment.py').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / 'src/experiment.py').exists(), 'Buka notebook dari folder proyek FoodSave.'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import json
import numpy as np
import pandas as pd
from IPython.display import display, Image, Markdown, Code
from src.experiment import OUT, ART, load_data, prepare, run_model, evaluate
pd.set_option('display.max_columns', 40)
pd.set_option('display.precision', 4)
get_ipython().run_line_magic('matplotlib', 'inline')


In [2]:
import matplotlib.pyplot as plt
df=load_data()
train=df[df.split=='train'].copy()
display(train.head())
display(train.isna().sum().to_frame('jumlah kosong'))

,ID,date,meals_served,kitchen_staff,temperature_C,humidity_percent,day_of_week,special_event,past_waste_kg,staff_experience,waste_category,food_waste_kg,_row,split
0,397,2022-01-01,202,9,31.9768,66.8266,5,0,45.4455,Intermediate,MEAT,39.9011,341,train
1,427,2022-01-01,202,9,32.3188,66.8266,5,0,45.4455,Intermediate,MEAT,39.9011,364,train
2,201,2022-01-02,448,16,10.0925,56.1303,6,0,29.8900,intermediate,GRAINS,53.2034,171,train
3,983,2022-01-02,448,16,10.1286,56.1303,6,0,29.8900,intermediate,GRAINS,53.2034,849,train
4,162,2022-01-03,370,14,30.4987,64.6350,0,0,41.5516,Beginner,Vegetables,61.3754,141,train


,jumlah kosong
ID,0
date,0
meals_served,0
kitchen_staff,0
temperature_C,0
humidity_percent,0
day_of_week,0
special_event,0
past_waste_kg,0
staff_experience,106


## Distribusi limbah dan urutan tanggal
Histogram menunjukkan jumlah kemunculan tiap rentang limbah. Grafik waktu memperlihatkan rata-rata observasi per tanggal, bukan total limbah seluruh restoran.

In [3]:
fig, axes=plt.subplots(1,2,figsize=(12,4))
axes[0].hist(train.food_waste_kg,bins=30); axes[0].set(xlabel='Limbah (kg)',ylabel='Jumlah observasi',title='Distribusi target training')
train.groupby('date').food_waste_kg.mean().plot(ax=axes[1],title='Rata-rata limbah per tanggal training')
fig.tight_layout(); display(fig); plt.close(fig)

## Contoh variasi format dan korelasi
Korelasi hanya hubungan statistik. Korelasi tinggi tidak membuktikan bahwa mengubah satu input pasti mengubah limbah.

In [4]:
for col in ['meals_served','kitchen_staff','staff_experience','waste_category']:
    print(col, ':', train[col].astype(str).unique()[:20])
numeric=train.select_dtypes(include='number').drop(columns=['ID','_row'],errors='ignore')
display(numeric.corr().round(3))

meals_served : <StringArray>
[ '202',  '448',  '370',  '206',  '171',  '288',  '120',  '314',  '430',
  '187',  '472', '4590',  '251', '4080',  '357',  '443',  '393',  '485',
  '291',  '376']
Length: 20, dtype: str
kitchen_staff : <StringArray>
[ '9', '16', '14', '11', '13',  '5', '10', '18',  '8',  '7', '17',  '6', '19',
 '12', '15']
Length: 15, dtype: str
staff_experience : <StringArray>
['Intermediate', 'intermediate', 'Beginner', nan, 'EXPERT']
Length: 5, dtype: str
waste_category : <StringArray>
['MEAT', 'GRAINS', 'Vegetables', 'MeAt', 'dairy']
Length: 5, dtype: str


,meals_served,kitchen_staff,temperature_C,humidity_percent,day_of_week,special_event,past_waste_kg,food_waste_kg
meals_served,1.000,-0.081,-0.037,-0.063,-0.021,0.014,-0.049,0.817
kitchen_staff,-0.081,1.000,-0.049,-0.044,0.060,-0.077,-0.081,-0.036
temperature_C,-0.037,-0.049,1.000,0.005,0.046,-0.013,0.018,-0.027
humidity_percent,-0.063,-0.044,0.005,1.000,0.017,0.008,0.059,-0.042
day_of_week,-0.021,0.060,0.046,0.017,1.000,-0.070,0.011,-0.030
special_event,0.014,-0.077,-0.013,0.008,-0.070,1.000,0.046,0.146
past_waste_kg,-0.049,-0.081,0.018,0.059,0.011,0.046,1.000,0.321
food_waste_kg,0.817,-0.036,-0.027,-0.042,-0.030,0.146,0.321,1.000


## Langkah tambahan 1 Membersihkan salinan training untuk eksplorasi
Salinan ini hanya membantu membaca pola. Pipeline pemodelan tetap dipasang ulang pada training dan tidak mengambil statistik dari test. Pada scatter, satu titik mewakili satu observasi, bukan satu restoran terverifikasi.

In [5]:
from src.experiment import FoodPreprocessor
prep=FoodPreprocessor('robust').fit(train)
view=prep.native(train)
view['food_waste_kg']=train.food_waste_kg.to_numpy()
view['date']=train.date.to_numpy()
fig,axes=plt.subplots(1,3,figsize=(15,4))
for ax,col in zip(axes,['meals_served','kitchen_staff','temperature_C']):
    ax.scatter(view[col],view.food_waste_kg,alpha=.5,s=14)
    ax.set(xlabel=col,ylabel='Limbah kg',title=f'Hubungan {col} dan limbah')
fig.tight_layout();display(fig);plt.close(fig)


## Langkah tambahan 2 Membaca hubungan angka secara hati-hati
Koefisien korelasi dekat +1 berarti dua variabel cenderung naik bersama, dekat −1 bergerak berlawanan, dan dekat 0 tidak menunjukkan hubungan linear kuat. Korelasi bisa dipengaruhi outlier atau hubungan fitur turunan; ini bukan bukti sebab-akibat. Tabel berikut menghindari ID sebagai fitur.

In [6]:
corr=view.select_dtypes(include='number').corr()
display(corr.food_waste_kg.drop('food_waste_kg').sort_values(ascending=False).to_frame('korelasi_dengan_target'))
fig,ax=plt.subplots(figsize=(10,8));im=ax.imshow(corr,vmin=-1,vmax=1,cmap='coolwarm')
ax.set_xticks(range(len(corr)),corr.columns,rotation=90);ax.set_yticks(range(len(corr)),corr.columns)
fig.colorbar(im,ax=ax,label='Korelasi');fig.tight_layout();display(fig);plt.close(fig)


,korelasi_dengan_target
meals_served,0.5968
portion_per_staff,0.4454
special_event_portion,0.2041
special_event,0.1460
month_cos,0.0530
is_weekend,0.0033
temperature_C,-0.0282
kitchen_staff,-0.0359
humidity_percent,-0.0421
month_sin,-0.0953


## Langkah tambahan 3 Acara khusus dan pengalaman staf
Rata-rata kelompok dapat berbeda karena jumlah porsi atau kondisi lain juga berbeda. Bandingkan jumlah observasi, median, dan variasi, bukan hanya rata-rata. Tidak ada uji kausal pada analisis ini.

In [7]:
for col in ['special_event','staff_experience']:
    display(view.groupby(col).food_waste_kg.agg(['count','mean','median','std']).round(3))
    groups=list(view.groupby(col))
    fig,ax=plt.subplots(figsize=(7,4))
    ax.boxplot([g.food_waste_kg.to_numpy() for _,g in groups],tick_labels=[str(k) for k,_ in groups])
    ax.set(xlabel=col,ylabel='Limbah kg',title=f'Distribusi limbah menurut {col}')
    fig.tight_layout();display(fig);plt.close(fig)


,count,mean,median,std
special_event,,,,
0.0,515,43.127,40.605,25.396
1.0,49,56.624,53.282,29.872


,count,mean,median,std
staff_experience,,,,
beginner,115,47.430,40.970,30.481
expert,113,46.380,42.594,31.733
intermediate,336,42.529,40.837,21.931


## Langkah tambahan 4 Menilai outlier tanpa membuang target
IQR adalah rentang antara kuartil 25% dan 75%. Nilai di luar Q1 − 1,5 IQR atau Q3 + 1,5 IQR ditandai ekstrem secara statistik. Ekstrem bukan berarti pasti salah. Tabel target hanya untuk diagnosis; target training/validasi/test tidak di-clipping oleh eksperimen.

In [8]:
outliers=[]
for col in ['meals_served','kitchen_staff','temperature_C','humidity_percent','food_waste_kg']:
    s=pd.to_numeric(train[col],errors='coerce');q1,q3=s.quantile([.25,.75]);iqr=q3-q1
    low,high=q1-1.5*iqr,q3+1.5*iqr
    outliers.append({'kolom':col,'batas_bawah':low,'batas_atas':high,'jumlah_ekstrem':int(((s<low)|(s>high)).sum())})
display(pd.DataFrame(outliers))


,kolom,batas_bawah,batas_atas,jumlah_ekstrem
0,meals_served,-83.5000,704.5000,12
1,kitchen_staff,-2.5000,25.5000,0
2,temperature_C,-3.6113,48.1653,16
3,humidity_percent,1.2913,120.6711,0
4,food_waste_kg,7.4971,75.8333,12


## Langkah tambahan 5 Variasi kalender dan perubahan di dalam training
Kita membandingkan dua bagian training untuk melihat indikasi perubahan distribusi tanpa mengintip test. Pola kalender tidak boleh disebut pola satu restoran karena identitas restoran belum tersedia. Bulan dengan data sedikit dapat menghasilkan rata-rata yang tidak stabil.

In [9]:
calendar=train.assign(hari=train.date.dt.dayofweek,bulan=train.date.dt.month)
display(calendar.groupby('hari').food_waste_kg.agg(['count','mean','median']))
display(calendar.groupby('bulan').food_waste_kg.agg(['count','mean','median']))
middle=len(view)//2
display(pd.DataFrame({'median_training_awal':view.iloc[:middle].select_dtypes('number').median(),'median_training_akhir':view.iloc[middle:].select_dtypes('number').median()}))


C:\Users\HP\AppData\Local\Temp\ipykernel_6052\2145113382.py:1: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  calendar=train.assign(hari=train.date.dt.dayofweek,bulan=train.date.dt.month)


,count,mean,median
hari,,,
0,80,43.8806,43.6313
1,75,43.7255,42.2332
2,82,50.9677,40.8758
3,78,43.9659,42.7031
4,82,38.6154,37.5492
5,84,45.9090,42.0876
6,83,42.9356,43.0701


,count,mean,median
bulan,,,
1,81,50.4866,40.6049
2,63,40.2675,39.3309
3,55,42.8569,42.1180
4,54,39.0829,37.2084
5,57,39.9911,41.7375
6,52,39.8248,39.6232
7,57,47.0980,44.1570
8,42,53.2994,45.1309
9,25,46.3487,42.7688


,median_training_awal,median_training_akhir
meals_served,3.1450e+02,2.9050e+02
kitchen_staff,1.2000e+01,1.2000e+01
temperature_C,2.2041e+01,2.2078e+01
humidity_percent,5.9716e+01,6.3720e+01
special_event,0.0000e+00,0.0000e+00
is_weekend,0.0000e+00,0.0000e+00
month_sin,5.0000e-01,1.2246e-16
month_cos,-1.8370e-16,6.1232e-17
staff_per_portion,3.8418e-02,3.8381e-02
portion_per_staff,2.6029e+01,2.6055e+01


## Insight EDA dan konsekuensi untuk pemodelan

In [10]:
top=corr.food_waste_kg.drop('food_waste_kg').abs().sort_values(ascending=False).head(3)
display(Markdown('Tiga hubungan linear terkuat pada salinan training: '+', '.join(f'**{k}** (|r|={v:.3f})' for k,v in top.items())+'. Hubungan ini perlu diuji melalui model dan tidak membuktikan pengaruh kausal.'))
display(Markdown('Nilai ekstrem tetap diuji secara jujur: Standard mempertahankan fitur ekstrem, Robust membatasi fitur memakai IQR training. Target evaluasi tetap aktual. Kita belum menyimpulkan Robust lebih baik sampai metrik validasi dihitung.'))


Tiga hubungan linear terkuat pada salinan training: **meals_served** (|r|=0.597), **portion_per_staff** (|r|=0.445), **staff_per_portion** (|r|=0.282). Hubungan ini perlu diuji melalui model dan tidak membuktikan pengaruh kausal.

Nilai ekstrem tetap diuji secara jujur: Standard mempertahankan fitur ekstrem, Robust membatasi fitur memakai IQR training. Target evaluasi tetap aktual. Kita belum menyimpulkan Robust lebih baik sampai metrik validasi dihitung.